In [2]:
!pip install folktables ucimlrepo fairlearn

In [3]:
import subprocess
subprocess.run(["pip", "install", "folktables", "ucimlrepo", "fairlearn", "--quiet",
                "--break-system-packages"], capture_output=True)

import os, gc, json, time, copy, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

WORK_DIR = "/kaggle/working"
PLOT_DIR = os.path.join(WORK_DIR, "agad_plots_v8")
os.makedirs(PLOT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SEEDS_FINAL  = [42, 7, 123]           # 3 seeds — architecture is simpler now
SEEDS_FINAL = [ 42 ]
RUN_DATASETS = ["adult", "acs_income", "acs_employment", "communities_crime"]

HIDDEN_DIM   = 128
DROPOUT      = 0.25
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 35
GRL_MAX      = 1.0
ADV_STEPS    = 3
ADV_LR_MULT  = 2.0
LAMBDA_MAX   = 2.0
PHASE1_END   = 10
PHASE2_END   = 20
MIN_SG_N     = 40
MIN_SG_FRAC  = 0.03
MAX_SG_FRAC  = 0.50
MAX_DEPTH    = 3
TOP_K_REPORT = 5
MIN_POS_FOR_EO      = 5
AUC_PROXY_SHARPNESS = 10.0
AUC_PROXY_SUBSAMPLE = 64

FULL_CFG = dict(
    adult             = dict(epochs=80,  batch=2048),
    acs_income        = dict(epochs=120, batch=4096),
    acs_employment    = dict(epochs=140, batch=4096),
    communities_crime = dict(epochs=130, batch=256),
)

VANILLA_ACC = dict(
    adult             = 0.7886,
    acs_income        = 0.7978,
    acs_employment    = 0.8075,
    communities_crime = 0.7118,
)
ACC_TOLERANCE = 0.05

# SGPen fully removed — no gamma_sg, no FrozenMaskManager, no penalty term anywhere
BEST_PARAMS = {
    "adult": {
        "eo": dict(lambda0=0.598, alpha=14.48, task_weight=1.85, top_k_loss=3, sg_warmup=10),
        "dp": dict(lambda0=0.65,  alpha=14.48, task_weight=1.85, top_k_loss=3, sg_warmup=10),
    },
    "acs_income": {
        "eo": dict(lambda0=0.55, alpha=15.0, task_weight=1.4,  top_k_loss=4, sg_warmup=10),
        "dp": dict(lambda0=0.74, alpha=13.0, task_weight=1.68, top_k_loss=3, sg_warmup=10),
    },
    "acs_employment": {
        "eo": dict(lambda0=0.75, alpha=12.0, task_weight=1.8,  top_k_loss=4, sg_warmup=3),
        "dp": dict(lambda0=0.40, alpha=10.0, task_weight=1.55, top_k_loss=3, sg_warmup=15),
    },
    # Communities & Crime: boosted lambda0/alpha for EO to push onto Pareto frontier
    "communities_crime": {
        "eo": dict(lambda0=0.72, alpha=16.0,  task_weight=1.92, top_k_loss=3, sg_warmup=10),
        "dp": dict(lambda0=0.80, alpha=17.0,  task_weight=1.8,  top_k_loss=3, sg_warmup=12),
    },
}

FG_CKPT_WEIGHT = {
    "adult":             {"eo": 0.0,  "dp": 0.0},
    "acs_income":        {"eo": 0.3,  "dp": 0.0},
    "acs_employment":    {"eo": 0.0,  "dp": 0.0},
    "communities_crime": {"eo": 0.0,  "dp": 0.0},
}

MIN_CKPT_EPOCH = {
    "adult":             {"eo": 0,  "dp": 0},
    "acs_income":        {"eo": 10, "dp": 10},
    "acs_employment":    {"eo": 15, "dp": 15},
    "communities_crime": {"eo": 0,  "dp": 0},
}

SCORE_MODE = {
    "adult":             {"eo": "with_auc", "dp": "with_auc"},
    "acs_income":        {"eo": "with_auc", "dp": "with_auc"},
    "acs_employment":    {"eo": "with_auc", "dp": "with_auc"},
    "communities_crime": {"eo": "with_auc", "dp": "with_auc"},
}

PR_ETA_BEST = {
    "adult":             {"eo": 2.0,  "dp": 5.0},
    "acs_income":        {"eo": 5.0,  "dp": 5.0},
    "acs_employment":    {"eo": 2.0,  "dp": 2.0},
    "communities_crime": {"eo": 1.0,  "dp": 2.0},
}
PR_ACC_PENALTY_COEF = 3.0

# ── Colour palette — SGPen colours REMOVED ──────────────────────────────────
PALETTE = {
    "vanilla"          : "#6c757d",
    "adv_eo"           : "#4e9af1",
    "adv_dp"           : "#1a6fc4",
    "fairlearn_eo"     : "#b07ded",
    "fairlearn_dp"     : "#7c3aed",
    "prejudice_rem_eo" : "#f9a8d4",
    "prejudice_rem_dp" : "#db2777",
    "agad_eo_tuned"    : "#2dc653",
    "agad_dp_tuned"    : "#1a7c34",
    "abl_no_auditor_eo": "#fde68a",
    "abl_no_auditor_dp": "#d97706",
}

METHOD_LABELS = {
    "vanilla"          : "Vanilla NN",
    "adv_eo"           : "Adv-EO",
    "adv_dp"           : "Adv-DP",
    "fairlearn_eo"     : "FL-AdvDeb-EO",
    "fairlearn_dp"     : "FL-AdvDeb-DP",
    "prejudice_rem_eo" : "PrejRem-EO",
    "prejudice_rem_dp" : "PrejRem-DP",
    "agad_eo_tuned"    : "AGAD-EO ★",
    "agad_dp_tuned"    : "AGAD-DP ★",
    "abl_no_auditor_eo": "No-Auditor-EO",
    "abl_no_auditor_dp": "No-Auditor-DP",
}

DS_LABELS = {
    "adult"            : "Adult Income",
    "acs_income"       : "ACS Income",
    "acs_employment"   : "ACS Employment",
    "communities_crime": "Communities & Crime",
}

# SGPen removed from COMPARISON_METHODS
COMPARISON_METHODS = [
    "vanilla", "adv_eo", "adv_dp",
    "fairlearn_eo", "fairlearn_dp",
    "prejudice_rem_eo", "prejudice_rem_dp",
    "agad_eo_tuned", "agad_dp_tuned",
]

# Three clean ablation conditions: vanilla, GRL-only, full AGAD (Auditor+GRL)
ABLATION_EO_METHODS = ["agad_eo_tuned", "abl_no_auditor_eo"]
ABLATION_DP_METHODS = ["agad_dp_tuned", "abl_no_auditor_dp"]

ABLATION_LABELS = {
    "agad_eo_tuned"    : "Full AGAD\n(Auditor+GRL)",
    "agad_dp_tuned"    : "Full AGAD\n(Auditor+GRL)",
    "abl_no_auditor_eo": "GRL only\n(no Auditor)",
    "abl_no_auditor_dp": "GRL only\n(no Auditor)",
}

print("=" * 70)
print("  AGAD v8 — SGPen fully removed; 3-condition ablation; clean tables")
print("=" * 70)
print(f"  Device     : {DEVICE}")
print(f"  Seeds      : {SEEDS_FINAL}")
print(f"  SGPen      : FULLY REMOVED (no baselines, no loss term, no params)")
print(f"  Ablations  : vanilla | GRL-only (no_auditor) | full AGAD (Auditor+GRL)")
print(f"  Output     : two tables per dataset — EO table and DP table, AGAD last row")
print(f"  Baselines  : all unconstrained — Pareto frontier reported at end")


# ════════════════════════════════════════════════════════════════════════════
#  UTILITIES
# ════════════════════════════════════════════════════════════════════════════

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def ensure_binary(sv):
    sv = np.asarray(sv).ravel(); u = np.unique(sv)
    if len(u) <= 1: return np.zeros(len(sv), int)
    if len(u) == 2: return (sv == u[1]).astype(int)
    maj = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != maj).astype(int)

def safe_auc(yt, yp):
    try:    return float(roc_auc_score(yt, yp)) if len(np.unique(yt)) >= 2 else float("nan")
    except: return float("nan")

def strat_split(X, y, sens_list, test_size=0.2, seed=42):
    key = np.array(y).astype(str)
    for s in sens_list:
        key = key + "_" + np.array(s).astype(str)
    u, c = np.unique(key, return_counts=True)
    key[np.isin(key, u[c < 2])] = np.array(y)[np.isin(key, u[c < 2])].astype(str)
    try:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=key, random_state=seed)
    except:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=y, random_state=seed)

def eo_gap(y_true, y_prob, mask):
    sg_t = y_true[mask]; sg_p = y_prob[mask]
    ot_t = y_true[~mask]; ot_p = y_prob[~mask]
    if len(np.unique(sg_t)) < 2 or len(np.unique(ot_t)) < 2: return 0.0
    tpr_sg = sg_p[sg_t == 1].mean() if (sg_t == 1).sum() > 0 else 0.0
    tpr_ot = ot_p[ot_t == 1].mean() if (ot_t == 1).sum() > 0 else 0.0
    fpr_sg = sg_p[sg_t == 0].mean() if (sg_t == 0).sum() > 0 else 0.0
    fpr_ot = ot_p[ot_t == 0].mean() if (ot_t == 0).sum() > 0 else 0.0
    return float(max(abs(tpr_sg - tpr_ot), abs(fpr_sg - fpr_ot)))

def dp_gap(y_prob, mask):
    return float(abs(y_prob[mask].mean() - y_prob.mean()))

def fg_metric(y_true, y_prob, sgs, k=TOP_K_REPORT, metric="eo"):
    gaps = []
    for sg in sgs:
        mem = sg["mem"]
        if len(np.unique(y_true[mem])) < 2: continue
        g = eo_gap(y_true, y_prob, mem) if metric == "eo" else dp_gap(y_prob, mem)
        gaps.append(g)
    if not gaps: return 0.0
    gaps.sort(reverse=True)
    return float(np.mean(gaps[:k]))

def get_depth_limit(epoch):
    if epoch < PHASE1_END: return 1
    if epoch < PHASE2_END: return 2
    return 3

def enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=2):
    n_attr = len(attr_names)
    min_n  = max(MIN_SG_N, int(MIN_SG_FRAC * n))
    max_n  = int(MAX_SG_FRAC * n)
    sgs, seen = [], set()
    for mask in range(1, 2 ** n_attr):
        active = [i for i in range(n_attr) if mask & (1 << i)]
        if len(active) > max_depth: continue
        for vals in range(2 ** len(active)):
            req = [(vals >> j) & 1 for j in range(len(active))]
            mem = np.ones(n, dtype=bool)
            for ai, rv in zip(active, req):
                mem &= (sv_bin_list[ai] == rv)
            key = mem.tobytes()
            if key in seen: continue
            seen.add(key)
            sz = int(mem.sum())
            if sz < min_n or sz > max_n: continue
            spec = [(attr_names[i], req[j]) for j, i in enumerate(active)]
            sgs.append({"mem": mem, "spec": spec})
    return sgs

def audit(epoch, y_val, p_val, sv_val_bin, attr_names, metric="eo"):
    depth  = min(get_depth_limit(epoch), MAX_DEPTH)
    n      = len(p_val)
    sgs    = enumerate_subgroups(sv_val_bin, attr_names, n, max_depth=depth)
    ranked = []
    for sg in sgs:
        mem = sg["mem"]
        if len(np.unique(y_val[mem])) < 2: continue
        gap = eo_gap(y_val, p_val, mem) if metric == "eo" else dp_gap(p_val, mem)
        ranked.append((gap, sg["spec"], mem))
    ranked.sort(key=lambda x: -x[0])
    worst_gap  = ranked[0][0] if ranked else 0.0
    worst_spec = ranked[0][1] if ranked else None
    topk_specs = [r[1] for r in ranked[:6]]
    fg_k       = float(np.mean([r[0] for r in ranked[:TOP_K_REPORT]])) if ranked else 0.0
    return float(worst_gap), worst_spec, float(fg_k), topk_specs

def adaptive_lambda(V_t, lambda0, alpha):
    return min(lambda0 * (1.0 + alpha * V_t), LAMBDA_MAX)

def build_intersection_labels(sv_bin_list, n_attrs):
    n      = len(sv_bin_list[0])
    labels = np.zeros(n, dtype=np.int64)
    for sv in sv_bin_list:
        labels = labels * 2 + np.asarray(sv, dtype=np.int64)
    return labels

def evaluate(proba, y, sv_bin_list, attr_names, tag="", verbose=True):
    pred  = (proba > 0.5).astype(int)
    acc   = float(accuracy_score(y, pred))
    auc   = safe_auc(y, proba)
    n     = len(proba)
    sgs   = enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=MAX_DEPTH)
    wc_eo = max((eo_gap(y, proba, sg["mem"])
                 for sg in sgs if len(np.unique(y[sg["mem"]])) >= 2), default=0.0)
    wc_dp = max((dp_gap(proba, sg["mem"]) for sg in sgs), default=0.0)
    fg_eo = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="eo")
    fg_dp = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="dp")
    if verbose:
        print(f"    [{tag:<36}]  WC-EO={wc_eo:.4f}  WC-DP={wc_dp:.4f}  "
              f"FG-EO={fg_eo:.4f}  FG-DP={fg_dp:.4f}  Acc={acc:.4f}  AUC={auc:.4f}")
    return {"wc_eo": wc_eo, "wc_dp": wc_dp, "fg_eo": fg_eo,
            "fg_dp": fg_dp, "acc": acc, "auc": auc}

def aggregate_seeds(results_list):
    keys = ["wc_eo", "wc_dp", "fg_eo", "fg_dp", "acc", "auc"]
    agg  = {}
    for k in keys:
        vals = [r[k] for r in results_list if k in r and r[k] is not None
                and not (isinstance(r[k], float) and np.isnan(r[k]))]
        agg[k]          = float(np.mean(vals)) if vals else float("nan")
        agg[k + "_std"] = float(np.std(vals))  if len(vals) > 1 else 0.0
        agg[k + "_all"] = [float(v) for v in vals]
    return agg

def pareto_frontier(results_dict, fairness_key, acc_key):
    methods = list(results_dict.keys())
    frontier = []
    for m in methods:
        f_m   = results_dict[m].get(fairness_key, float("nan"))
        acc_m = results_dict[m].get(acc_key, float("nan"))
        if np.isnan(f_m) or np.isnan(acc_m):
            continue
        dominated = False
        for m2 in methods:
            if m2 == m: continue
            f_m2   = results_dict[m2].get(fairness_key, float("nan"))
            acc_m2 = results_dict[m2].get(acc_key, float("nan"))
            if np.isnan(f_m2) or np.isnan(acc_m2):
                continue
            if f_m2 <= f_m and acc_m2 >= acc_m and (f_m2 < f_m or acc_m2 > acc_m):
                dominated = True
                break
        if not dominated:
            frontier.append(m)
    return frontier

def print_section(title, width=70):
    print(); print("=" * width); print(f"  {title}"); print("=" * width)

def print_subsection(title, width=70):
    print(); print("-" * width); print(f"  {title}"); print("-" * width)


# ════════════════════════════════════════════════════════════════════════════
#  TWO-TABLE SUMMARY PRINTER  (EO table + DP table, AGAD as last rows)
# ════════════════════════════════════════════════════════════════════════════

def print_two_tables(ds_key, ds_results):
    """
    Print two side-by-side tables for a dataset:
      Table 1 — EO metrics: WC-EO, FG-EO, Acc, AUC
      Table 2 — DP metrics: WC-DP, FG-DP, Acc, AUC
    Rows ordered: baselines first, then AGAD variants last for easy comparison.
    """
    row_order = [
        "vanilla",
        "adv_eo", "adv_dp",
        "fairlearn_eo", "fairlearn_dp",
        "prejudice_rem_eo", "prejudice_rem_dp",
        "abl_no_auditor_eo", "abl_no_auditor_dp",
        "agad_eo_tuned", "agad_dp_tuned",           # AGAD last
    ]
    # Keep only rows that exist in this ds_results
    rows = [m for m in row_order if m in ds_results]

    def cell(r, k):
        v = r.get(k, float("nan")); s = r.get(k + "_std", 0.0)
        if np.isnan(v): return "  nan   "
        return f"{v:.4f}±{s:.4f}"

    sep_eo = "─" * 72
    sep_dp = "─" * 72

    # ── EO Table ──
    print(f"\n  {'─'*72}")
    print(f"  EO TABLE  ({DS_LABELS[ds_key]})")
    print(f"  {'─'*72}")
    print(f"  {'Method':<28}  {'WC-EO':>18}  {'FG-EO':>18}  {'Acc':>10}  {'AUC':>10}")
    print(f"  {sep_eo}")
    for m in rows:
        r    = ds_results[m]
        flag = " ←" if "agad" in m else ""
        print(f"  {METHOD_LABELS.get(m,m):<28}  "
              f"{cell(r,'wc_eo'):>18}  "
              f"{cell(r,'fg_eo'):>18}  "
              f"{cell(r,'acc'):>10}  "
              f"{cell(r,'auc'):>10}"
              f"{flag}")
    print(f"  {sep_eo}")

    # ── DP Table ──
    print(f"\n  {'─'*72}")
    print(f"  DP TABLE  ({DS_LABELS[ds_key]})")
    print(f"  {'─'*72}")
    print(f"  {'Method':<28}  {'WC-DP':>18}  {'FG-DP':>18}  {'Acc':>10}  {'AUC':>10}")
    print(f"  {sep_dp}")
    for m in rows:
        r    = ds_results[m]
        flag = " ←" if "agad" in m else ""
        print(f"  {METHOD_LABELS.get(m,m):<28}  "
              f"{cell(r,'wc_dp'):>18}  "
              f"{cell(r,'fg_dp'):>18}  "
              f"{cell(r,'acc'):>10}  "
              f"{cell(r,'auc'):>10}"
              f"{flag}")
    print(f"  {sep_dp}")


# ════════════════════════════════════════════════════════════════════════════
#  MODEL COMPONENTS
# ════════════════════════════════════════════════════════════════════════════

class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha; return x.clone()
    @staticmethod
    def backward(ctx, g):
        return -ctx.alpha * g, None

class GradReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__(); self.alpha = alpha
    def forward(self, x):
        return GRL.apply(x, self.alpha)

class Encoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        h = HIDDEN_DIM
        self.rep_dim = h // 2 + 32
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),         nn.BatchNorm1d(h),            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(h, 256),            nn.BatchNorm1d(256),          nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, 128),          nn.BatchNorm1d(128),          nn.GELU(), nn.Dropout(DROPOUT * 0.5),
            nn.Linear(128, self.rep_dim), nn.BatchNorm1d(self.rep_dim), nn.GELU(), nn.Dropout(DROPOUT * 0.5),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x)

class PredictorHead(nn.Module):
    def __init__(self, rep_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, rep_dim // 2), nn.GELU(),
            nn.Linear(rep_dim // 2, 1),
        )
    def forward(self, z): return self.net(z).squeeze(1)

class IntersectionAdversaryHead(nn.Module):
    def __init__(self, rep_dim, n_marginal_attrs, n_intersection_classes):
        super().__init__()
        h = HIDDEN_DIM // 2
        self.grl   = GradReversal(1.0)
        self.trunk = nn.Sequential(
            nn.Linear(rep_dim, h), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(h, h // 2), nn.GELU(),
        )
        self.marginal_out     = nn.Linear(h // 2, n_marginal_attrs)
        self.intersection_out = nn.Linear(h // 2, n_intersection_classes)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def set_alpha(self, a): self.grl.alpha = a

    def forward(self, z, alpha=None):
        if alpha is not None: self.grl.alpha = alpha
        h = self.trunk(self.grl(z))
        return self.marginal_out(h), self.intersection_out(h)


# ════════════════════════════════════════════════════════════════════════════
#  CORE AGAD TRAINING LOOP  (v8 — SGPen fully removed, no gamma_sg anywhere)
#
#  disable_auditor : bool
#      Disables subgroup discovery. V_t=0, lambda fixed at lambda0.
#      GRL still runs normally.  => "GRL only" / "no_auditor" ablation.
# ════════════════════════════════════════════════════════════════════════════

def run_agad_with_hparams(d, cfg, metric, hp, seed=42,
                           verbose=False, epoch_verbose=False,
                           score_mode="with_auc",
                           fg_ckpt_weight=0.0,
                           min_ckpt_epoch=0,
                           disable_auditor=False):
    set_seed(seed)

    lambda0     = hp["lambda0"]
    alpha_val   = hp["alpha"]
    task_weight = hp["task_weight"]

    n_attrs = len(d["attr_names"])
    n_inter = 2 ** n_attrs

    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    inter_labels_np = build_intersection_labels(d["sv_tr"], n_attrs)
    inter_t  = torch.tensor(inter_labels_np, dtype=torch.long, device=DEVICE)
    idx_t    = torch.arange(len(d["y_tr"]), dtype=torch.long)

    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    adv  = IntersectionAdversaryHead(enc.rep_dim, n_attrs, n_inter).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    ce   = nn.CrossEntropyLoss()
    eps  = torch.tensor(1e-4, device=DEVICE)

    opt_enc = optim.AdamW(
        list(enc.parameters()) + list(head.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    opt_adv = optim.AdamW(adv.parameters(),
                          lr=LR * ADV_LR_MULT, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(
        opt_enc, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(
        TensorDataset(Xt, yt, idx_t),
        batch_size=cfg["batch"], shuffle=True, drop_last=True)

    best_score = np.inf
    best_enc   = best_head = None
    no_imp     = 0
    V_t        = 0.0
    acc_floor  = VANILLA_ACC[d["ds_key"]] - ACC_TOLERANCE

    for epoch in range(cfg["epochs"]):
        enc.eval(); head.eval()
        with torch.no_grad():
            pv_scan = torch.sigmoid(head(enc(Xv))).cpu().numpy()

        if disable_auditor:
            # GRL-only condition: no subgroup discovery, lambda fixed
            V_t   = 0.0
            lam_t = lambda0
            fg_k  = 0.0
        else:
            V_t, worst_spec, fg_k, topk_specs = audit(
                epoch, d["y_val"], pv_scan, d["sv_val"], d["attr_names"], metric=metric)
            lam_t = adaptive_lambda(V_t, lambda0, alpha_val)

        grl_alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
        adv.set_alpha(grl_alpha)
        enc.train(); head.train(); adv.train()

        for xb, yb, bidx in loader:

            # Adversary update (maximise sensitive attribute prediction)
            z_d = enc(xb).detach()
            for _ in range(ADV_STEPS):
                opt_adv.zero_grad(set_to_none=True)
                m_out, i_out = adv(z_d, alpha=0)
                loss_a = (
                    sum(nn.functional.binary_cross_entropy_with_logits(
                        m_out[:, i], sv_t[i][bidx])
                        for i in range(n_attrs)) / n_attrs
                    + ce(i_out, inter_t[bidx]))
                loss_a.backward()
                nn.utils.clip_grad_norm_(adv.parameters(), 1.0)
                opt_adv.step()

            opt_enc.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            prob   = torch.sigmoid(logits)
            task_loss = bce(logits, yb)

            # GRL fairness loss — pure Auditor+GRL, no SGPen term at all
            fair_loss = torch.tensor(0.0, device=DEVICE)
            m_adv, i_adv = adv(z)
            for i, sv in enumerate(sv_t):
                tgt = sv[bidx]
                if metric == "eo":
                    sf = tgt.float(); yf = yb.float()
                    g0y1 = (1 - sf) * yf;       g1y1 = sf * yf
                    g0y0 = (1 - sf) * (1 - yf); g1y0 = sf * (1 - yf)
                    if all(g.sum() > 1e-6 for g in [g0y1, g1y1, g0y0, g1y0]):
                        tpr0 = (prob * g0y1).sum() / (g0y1.sum() + eps)
                        tpr1 = (prob * g1y1).sum() / (g1y1.sum() + eps)
                        fpr0 = (prob * g0y0).sum() / (g0y0.sum() + eps)
                        fpr1 = (prob * g1y0).sum() / (g1y0.sum() + eps)
                        fair_loss += torch.max(
                            torch.abs(tpr0 - tpr1), torch.abs(fpr0 - fpr1))
                else:
                    n0 = (1 - tgt).sum() + eps; n1 = tgt.sum() + eps
                    fair_loss += torch.abs(
                        (prob * (1 - tgt)).sum() / n0
                        - (prob * tgt).sum() / n1)
                fair_loss += nn.functional.binary_cross_entropy_with_logits(
                    m_adv[:, i], tgt)
            fair_loss += ce(i_adv, inter_t[bidx])

            # Total loss: task + adaptive-lambda * GRL fairness — that's it
            loss = (task_weight * task_loss
                    + lam_t * fair_loss / max(n_attrs, 1))

            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt_enc.step()

        sched.step()

        enc.eval(); head.eval()
        with torch.no_grad():
            pv  = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        acc = accuracy_score(d["y_val"], (pv > 0.5).astype(int))
        auc = safe_auc(d["y_val"], pv)

        score = (V_t
                 + fg_ckpt_weight * fg_k
                 + 0.5 * max(0.0, acc_floor - acc))

        if score < best_score and epoch >= min_ckpt_epoch:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            no_imp     = 0
        elif epoch >= min_ckpt_epoch:
            no_imp += 1

        if epoch < min_ckpt_epoch:
            best_enc  = copy.deepcopy(enc.state_dict())
            best_head = copy.deepcopy(head.state_dict())

        if epoch_verbose and (epoch % 10 == 0 or epoch == cfg["epochs"] - 1):
            print(f"      epoch {epoch:03d}  V_t={V_t:.4f}  "
                  f"lam={lam_t:.3f}  "
                  f"acc={acc:.4f}  auc={auc:.4f}  "
                  f"score={score:.5f}  no_imp={no_imp}")

        if no_imp >= PATIENCE:
            if epoch_verbose:
                print(f"      [early stop at epoch {epoch}]")
            break

    enc.load_state_dict(best_enc)
    head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()

    tag = f"AGAD-{metric.upper()} (final)"
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"],
                    tag=tag, verbose=verbose)


# ════════════════════════════════════════════════════════════════════════════
#  VANILLA BASELINE
# ════════════════════════════════════════════════════════════════════════════

def run_vanilla(d, cfg, seed=42):
    set_seed(seed)
    Xt  = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt  = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv  = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    opt  = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    for epoch in range(cfg["epochs"]):
        enc.train(); head.train()
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            bce(head(enc(xb)), yb).backward()
            opt.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"])
        score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
        if score < best_score:
            best_score = score; best_enc = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict()); no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag="Vanilla NN")


# ════════════════════════════════════════════════════════════════════════════
#  ADV BASELINE  (static GRL, no SGPen — SGPen fully removed from codebase)
# ════════════════════════════════════════════════════════════════════════════

def _adv_static(d, cfg, metric, seed=42):
    """Pure adversarial debiasing baseline — no subgroup penalty, no auditor."""
    set_seed(seed)
    n_attrs = len(d["attr_names"])
    n_inter = 2 ** n_attrs
    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    inter_labels_np = build_intersection_labels(d["sv_tr"], n_attrs)
    inter_t = torch.tensor(inter_labels_np, dtype=torch.long, device=DEVICE)
    idx_t   = torch.arange(len(d["y_tr"]), dtype=torch.long)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    adv  = IntersectionAdversaryHead(enc.rep_dim, n_attrs, n_inter).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss(); ce = nn.CrossEntropyLoss()
    eps  = torch.tensor(1e-4, device=DEVICE)
    opt_enc = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                          lr=LR, weight_decay=WEIGHT_DECAY)
    opt_adv = optim.AdamW(adv.parameters(), lr=LR * ADV_LR_MULT, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt_enc, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt, idx_t),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    for epoch in range(cfg["epochs"]):
        alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
        adv.set_alpha(alpha); enc.train(); head.train(); adv.train()
        for xb, yb, bidx in loader:
            z_d = enc(xb).detach()
            for _ in range(ADV_STEPS):
                opt_adv.zero_grad(set_to_none=True)
                m_out, i_out = adv(z_d, alpha=0)
                la = (sum(nn.functional.binary_cross_entropy_with_logits(
                          m_out[:, i], sv_t[i][bidx]) for i in range(n_attrs)) / n_attrs
                      + ce(i_out, inter_t[bidx]))
                la.backward(); nn.utils.clip_grad_norm_(adv.parameters(), 1.0); opt_adv.step()
            opt_enc.zero_grad(set_to_none=True)
            z = enc(xb); logits = head(z); prob = torch.sigmoid(logits)
            task_loss = bce(logits, yb)
            fair_loss = torch.tensor(0.0, device=DEVICE)
            m_adv, i_adv = adv(z)
            for i, sv in enumerate(sv_t):
                tgt = sv[bidx]
                if metric == "eo":
                    sf = tgt.float(); yf = yb.float()
                    g0y1=(1-sf)*yf; g1y1=sf*yf; g0y0=(1-sf)*(1-yf); g1y0=sf*(1-yf)
                    if all(g.sum() > 1e-6 for g in [g0y1, g1y1, g0y0, g1y0]):
                        tpr0=(prob*g0y1).sum()/(g0y1.sum()+eps); tpr1=(prob*g1y1).sum()/(g1y1.sum()+eps)
                        fpr0=(prob*g0y0).sum()/(g0y0.sum()+eps); fpr1=(prob*g1y0).sum()/(g1y0.sum()+eps)
                        fair_loss += torch.max(torch.abs(tpr0-tpr1), torch.abs(fpr0-fpr1))
                else:
                    n0=(1-tgt).sum()+eps; n1=tgt.sum()+eps
                    fair_loss += torch.abs((prob*(1-tgt)).sum()/n0 - (prob*tgt).sum()/n1)
                fair_loss += nn.functional.binary_cross_entropy_with_logits(m_adv[:, i], tgt)
            fair_loss += ce(i_adv, inter_t[bidx])
            # No SGPen term — pure adversarial only
            loss = (2.0 * task_loss + 0.3 * fair_loss / n_attrs)
            loss.backward()
            nn.utils.clip_grad_norm_(list(enc.parameters()) + list(head.parameters()), 0.5)
            opt_enc.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"], metric=metric)
        score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
        if score < best_score:
            best_score = score; best_enc = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict()); no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    tag = f"Adv-{metric.upper()} (static)"
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag=tag)


# ════════════════════════════════════════════════════════════════════════════
#  FAIRLEARN
# ════════════════════════════════════════════════════════════════════════════

def run_fairlearn_adv(d, cfg, metric, seed=42):
    from fairlearn.adversarial import AdversarialFairnessClassifier
    set_seed(seed)
    X_tr_np = d["X_tr"].astype(np.float32)
    y_tr_np = d["y_tr"].astype(int)
    X_te_np = d["X_te"].astype(np.float32)
    y_te_np = d["y_te"].astype(int)
    sf_tr   = d["sv_tr"][0].astype(int)
    constraint = "equalized_odds" if metric == "eo" else "demographic_parity"
    mitigator = AdversarialFairnessClassifier(
        backend="torch",
        predictor_model=[128, "relu", 64, "relu"],
        adversary_model=[64, "relu"],
        constraints=constraint,
        epochs=min(cfg["epochs"], 50),
        batch_size=min(cfg["batch"], 2048),
        random_state=seed,
        progress_updates=None,
    )
    mitigator.fit(X_tr_np, y_tr_np, sensitive_features=sf_tr)
    proba = None
    for getter in ["predict_proba", "decision_function"]:
        fn = getattr(mitigator, getter, None)
        if fn is not None:
            try:
                raw = fn(X_te_np)
                if raw.ndim == 2 and raw.shape[1] == 2:
                    proba = raw[:, 1].astype(np.float64)
                else:
                    proba = raw.ravel().astype(np.float64)
                lo, hi = proba.min(), proba.max()
                if hi > lo:
                    proba = (proba - lo) / (hi - lo)
                break
            except Exception:
                pass
    if proba is None:
        hard = mitigator.predict(X_te_np).astype(np.float64)
        proba = hard + np.random.default_rng(seed).uniform(-1e-4, 1e-4, size=hard.shape)
        proba = np.clip(proba, 0.0, 1.0)
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    tag   = f"FL-AdvDeb-{metric.upper()}"
    return evaluate(proba, y_te_np, d["sv_te"], d["attr_names"], tag=tag, verbose=True)


# ════════════════════════════════════════════════════════════════════════════
#  PREJUDICE REMOVER  (reported straight — all unconstrained, Pareto framing)
# ════════════════════════════════════════════════════════════════════════════

def _prejudice_remover_loss(prob, y, sv_list, metric, eta, eps=1e-6):
    mi_total = torch.tensor(0.0, device=prob.device)
    for sv in sv_list:
        sv = sv.float()
        p_y1 = prob.mean().clamp(eps, 1 - eps)
        p_y0 = (1 - prob).mean().clamp(eps, 1 - eps)
        p_s1 = sv.mean().clamp(eps, 1 - eps)
        p_s0 = (1 - sv).mean().clamp(eps, 1 - eps)
        p_y1_s1 = (prob * sv).mean().clamp(eps, 1 - eps)
        p_y1_s0 = (prob * (1 - sv)).mean().clamp(eps, 1 - eps)
        p_y0_s1 = ((1 - prob) * sv).mean().clamp(eps, 1 - eps)
        p_y0_s0 = ((1 - prob) * (1 - sv)).mean().clamp(eps, 1 - eps)
        if metric == "eo":
            pos_mask = (y == 1)
            if pos_mask.sum() < 2: continue
            prob_pos = prob[pos_mask]; sv_pos = sv[pos_mask]
            p_y1_s1g = (prob_pos * sv_pos).mean().clamp(eps, 1 - eps)
            p_y1_s0g = (prob_pos * (1 - sv_pos)).mean().clamp(eps, 1 - eps)
            p_s1g    = sv_pos.mean().clamp(eps, 1 - eps)
            p_s0g    = (1 - sv_pos).mean().clamp(eps, 1 - eps)
            mi_total += (
                p_y1_s1g * torch.log(p_y1_s1g / (p_y1_s1g.detach() * p_s1g + eps)) +
                p_y1_s0g * torch.log(p_y1_s0g / (p_y1_s0g.detach() * p_s0g + eps))
            )
        else:
            mi_total += (
                p_y1_s1 * torch.log(p_y1_s1 / (p_y1 * p_s1 + eps)) +
                p_y1_s0 * torch.log(p_y1_s0 / (p_y1 * p_s0 + eps)) +
                p_y0_s1 * torch.log(p_y0_s1 / (p_y0 * p_s1 + eps)) +
                p_y0_s0 * torch.log(p_y0_s0 / (p_y0 * p_s0 + eps))
            )
    return eta * mi_total / max(len(sv_list), 1)


def run_prejudice_remover(d, cfg, metric, seed=42):
    set_seed(seed)
    eta       = PR_ETA_BEST[d["ds_key"]][metric]
    acc_floor = VANILLA_ACC[d["ds_key"]] - ACC_TOLERANCE
    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    idx_t = torch.arange(len(d["y_tr"]), dtype=torch.long)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    opt  = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt, idx_t),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    for epoch in range(cfg["epochs"]):
        enc.train(); head.train()
        for xb, yb, bidx in loader:
            opt.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            prob   = torch.sigmoid(logits)
            sv_b   = [sv[bidx] for sv in sv_t]
            loss   = (bce(logits, yb)
                      + _prejudice_remover_loss(prob, yb.long(), sv_b, metric, eta))
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv  = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        acc = accuracy_score(d["y_val"], (pv > 0.5).astype(int))
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"], metric=metric)
        score = (V
                 + 0.12 * (1 - auc if not np.isnan(auc) else 0)
                 + PR_ACC_PENALTY_COEF * max(0.0, acc_floor - acc))
        if score < best_score:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            no_imp     = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    tag = f"PrejRem-{metric.upper()} (eta={eta})"
    # Reported straight — no exclusion — Pareto framing handles tradeoff
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag=tag, verbose=True)


# ════════════════════════════════════════════════════════════════════════════
#  ABLATION RUNNER  (v8: single condition — no_auditor = GRL only)
# ════════════════════════════════════════════════════════════════════════════

def run_ablation(d, cfg, metric, ablation_type, seed=42):
    hp  = BEST_PARAMS[d["ds_key"]][metric]
    sm  = SCORE_MODE[d["ds_key"]][metric]
    fw  = FG_CKPT_WEIGHT[d["ds_key"]][metric]
    mce = MIN_CKPT_EPOCH[d["ds_key"]][metric]

    if ablation_type == "no_auditor":
        return run_agad_with_hparams(
            d, cfg, metric, hp, seed=seed,
            verbose=True, epoch_verbose=False,
            score_mode=sm, fg_ckpt_weight=fw, min_ckpt_epoch=mce,
            disable_auditor=True,
        )
    else:
        raise ValueError(f"Unknown ablation: {ablation_type}")


# ════════════════════════════════════════════════════════════════════════════
#  DATA LOADERS
# ════════════════════════════════════════════════════════════════════════════

def load_adult():
    from ucimlrepo import fetch_ucirepo
    repo  = fetch_ucirepo(id=2)
    X_df  = repo.data.features.copy()
    y_ser = repo.data.targets.copy()
    y_raw = y_ser.iloc[:, 0].astype(str).str.strip().str.rstrip(".")
    y     = (y_raw == ">50K").astype(int).values
    race_Black = (X_df["race"].astype(str).str.strip() == "Black").astype(int).values
    race_White = (X_df["race"].astype(str).str.strip() == "White").astype(int).values
    sex_Female = (X_df["sex"].astype(str).str.strip() == "Female").astype(int).values
    age_old    = (pd.to_numeric(X_df["age"], errors="coerce").fillna(0) >= 45).astype(int).values
    X_df = X_df.drop(columns=["race","sex","age","fnlwgt","education-num"], errors="ignore")
    X_df = pd.get_dummies(X_df)
    X    = X_df.values.astype(np.float32)
    valid = ~np.isnan(X).any(axis=1)
    X, y  = X[valid], y[valid]
    race_Black=race_Black[valid]; race_White=race_White[valid]
    sex_Female=sex_Female[valid]; age_old=age_old[valid]
    tr, te = strat_split(X, y, [race_Black, sex_Female, age_old])
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
    attr_names = ["race_Black","race_White","sex_Female","age_old"]
    sv_tr = [race_Black[tr], race_White[tr], sex_Female[tr], age_old[tr]]
    sv_te = [race_Black[te], race_White[te], sex_Female[te], age_old[te]]
    tr2, val = strat_split(X_tr, y[tr], sv_tr)
    return dict(ds_key="adult",
        X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
        X_te=X_te, y_te=y[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_acs_income():
    from folktables import ACSDataSource, ACSIncome
    ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs = ds.get_data(states=["CA"], download=True)
    features, label, group = ACSIncome.df_to_numpy(acs)
    acs_feature_cols = ["AGEP","COW","SCHL","MAR","OCCP","POBP","RELP","WKHP","SEX","RAC1P"]
    sex_idx  = acs_feature_cols.index("SEX"); race_idx = acs_feature_cols.index("RAC1P")
    age_idx  = acs_feature_cols.index("AGEP")
    sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]; age_col = features[:, age_idx]
    rW=(race_col==1).astype(int); rB=(race_col==2).astype(int)
    rA=(race_col==6).astype(int); sex=(sex_col==2).astype(int); age_old=(age_col>=45).astype(int)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    rW=rW[valid]; rB=rB[valid]; rA=rA[valid]; sex=sex[valid]; age_old=age_old[valid]
    tr, te = strat_split(features, label, [rW, rB, sex, age_old])
    sc = StandardScaler()
    X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
    sv_tr = [rW[tr], rB[tr], rA[tr], sex[tr], age_old[tr]]
    sv_te = [rW[te], rB[te], rA[te], sex[te], age_old[te]]
    attr_names = ["race_White","race_Black","race_Asian","sex_Female","age_old"]
    tr2, val = strat_split(X_tr, label[tr], sv_tr)
    return dict(ds_key="acs_income",
        X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
        X_te=X_te, y_te=label[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_acs_employment():
    from folktables import ACSDataSource, ACSEmployment
    ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs = ds.get_data(states=["CA"], download=True)
    features, label, _ = ACSEmployment.df_to_numpy(acs)
    acs_emp_cols = ["AGEP","SCHL","MAR","DIS","ESP","CIT","MIG","MIL","ANC",
                    "NATIVITY","DEAR","DEYE","DREM","SEX","RAC1P"]
    sex_idx  = acs_emp_cols.index("SEX"); race_idx = acs_emp_cols.index("RAC1P")
    age_idx  = acs_emp_cols.index("AGEP"); dis_idx  = acs_emp_cols.index("DIS")
    sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]
    age_col  = features[:, age_idx]; dis_col  = features[:, dis_idx]
    RACE_MAP = {1:0, 2:1, 3:3, 4:2, 5:2, 6:3, 7:3, 8:3, 9:3}
    race = np.array([RACE_MAP.get(int(r), 3) for r in race_col])
    sex  = (sex_col == 2).astype(int)
    rW=(race==0).astype(int); rB=(race==1).astype(int); rO=(race==3).astype(int)
    age_old=(age_col>=45).astype(int); disabled=(dis_col==1).astype(int)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    rW=rW[valid]; rB=rB[valid]; rO=rO[valid]; sex=sex[valid]
    age_old=age_old[valid]; disabled=disabled[valid]
    tr, te = strat_split(features, label, [rW, rB, sex, age_old, disabled])
    sc = StandardScaler()
    X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
    sv_tr = [rW[tr], rB[tr], rO[tr], sex[tr], age_old[tr], disabled[tr]]
    sv_te = [rW[te], rB[te], rO[te], sex[te], age_old[te], disabled[te]]
    attr_names = ["race_White","race_Black","race_Other","sex_Female","age_old","disabled"]
    tr2, val = strat_split(X_tr, label[tr], sv_tr)
    return dict(ds_key="acs_employment",
        X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
        X_te=X_te, y_te=label[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_communities_crime():
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=183)
    X_df = repo.data.features.copy()
    y_df = repo.data.targets.copy()
    y_cont = pd.to_numeric(y_df.iloc[:, 0], errors="coerce").values
    valid  = ~np.isnan(y_cont)
    y_cont = y_cont[valid]; X_df = X_df[valid].reset_index(drop=True)
    y = (y_cont > np.median(y_cont)).astype(int)
    def find_col(df, pats):
        for p in pats:
            m = [c for c in df.columns if p.lower() in c.lower()]
            if m: return pd.to_numeric(df[m[0]], errors="coerce")
        return None
    col_b = find_col(X_df, ["racepctblack","pctblack"])
    col_i = find_col(X_df, ["medIncome","medincome"])
    def binarise(col, high=True):
        if col is None: return np.zeros(len(y), int)
        c = col.fillna(col.median()).values
        return (c > np.median(c)).astype(int) if high else (c < np.median(c)).astype(int)
    s_race = binarise(col_b, high=True); s_inc = binarise(col_i, high=False)
    X_num = X_df.apply(pd.to_numeric, errors="coerce")
    X_num = X_num.dropna(axis=1, thresh=int(0.7*len(X_num))).fillna(X_num.median())
    drop_cols = [c for c in X_num.columns if any(p.lower() in c.lower()
                 for p in ["racepct","racePct","medIncome","ViolentCrimes","PctUnemployed"])]
    X = X_num.drop(columns=drop_cols, errors="ignore").values.astype(np.float32)
    tr, te = strat_split(X, y, [s_race, s_inc])
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
    sv_tr = [s_race[tr], s_inc[tr]]; sv_te = [s_race[te], s_inc[te]]
    attr_names = ["racial_composition","socioeconomic"]
    tr2, val = strat_split(X_tr, y[tr], sv_tr)
    return dict(ds_key="communities_crime",
        X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
        X_te=X_te, y_te=y[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

LOADERS = {
    "adult"            : load_adult,
    "acs_income"       : load_acs_income,
    "acs_employment"   : load_acs_employment,
    "communities_crime": load_communities_crime,
}


# ════════════════════════════════════════════════════════════════════════════
#  FINAL EVALUATION
# ════════════════════════════════════════════════════════════════════════════

print_section(f"FINAL EVALUATION  ({len(SEEDS_FINAL)} seeds: {SEEDS_FINAL})")

t_phase = time.time()
all_results = {}

for ds_key in RUN_DATASETS:
    print_subsection(f"Dataset: {ds_key.upper()}")
    cfg = FULL_CFG[ds_key]
    d   = LOADERS[ds_key]()
    print(f"  Train={len(d['y_tr'])}  Val={len(d['y_val'])}  Test={len(d['y_te'])}")

    ds_results = {}

    # ── Vanilla ───────────────────────────────────────────────────────────
    print(f"\n  -- vanilla --")
    seed_results = []
    for s in SEEDS_FINAL:
        print(f"    seed {s} ...", end=" ", flush=True)
        r = run_vanilla(d, cfg, seed=s)
        seed_results.append(r)
        print(f"WC-EO={r['wc_eo']:.4f}  WC-DP={r['wc_dp']:.4f}  "
              f"Acc={r['acc']:.4f}  AUC={r['auc']:.4f}")
    ds_results["vanilla"] = aggregate_seeds(seed_results)

    # ── Adv-EO / Adv-DP (pure adversarial, no SGPen) ──────────────────────
    for metric in ["eo", "dp"]:
        bname = f"adv_{metric}"
        print(f"\n  -- {bname} --")
        seed_results = []
        for s in SEEDS_FINAL:
            print(f"    seed {s} ...", end=" ", flush=True)
            r = _adv_static(d, cfg, metric, seed=s)
            seed_results.append(r)
            print(f"WC-EO={r['wc_eo']:.4f}  WC-DP={r['wc_dp']:.4f}  "
                  f"Acc={r['acc']:.4f}  AUC={r['auc']:.4f}")
        ds_results[bname] = aggregate_seeds(seed_results)

    # ── FairLearn ──────────────────────────────────────────────────────────
    for fl_metric in ["eo", "dp"]:
        fl_key = f"fairlearn_{fl_metric}"
        print(f"\n  -- {fl_key} --")
        fl_results = []
        for s in SEEDS_FINAL:
            print(f"    seed {s} ...")
            try:
                r = run_fairlearn_adv(d, cfg, fl_metric, seed=s)
            except Exception as e:
                print(f"    [{fl_key} seed {s} failed: {e}]")
                r = {k: float("nan") for k in
                     ["wc_eo","wc_dp","fg_eo","fg_dp","acc","auc"]}
            fl_results.append(r)
        ds_results[fl_key] = aggregate_seeds(fl_results)

    # ── Prejudice Remover (all methods unconstrained, Pareto framing) ──────
    for pr_metric in ["eo", "dp"]:
        pr_key = f"prejudice_rem_{pr_metric}"
        print(f"\n  -- {pr_key} --")
        pr_results = []
        for s in SEEDS_FINAL:
            print(f"    seed {s} ...")
            r = run_prejudice_remover(d, cfg, pr_metric, seed=s)
            pr_results.append(r)
        ds_results[pr_key] = aggregate_seeds(pr_results)

    # ── Full AGAD (Auditor+GRL) — AGAD rows come LAST ─────────────────────
    # Ablations run first so AGAD rows appear last in the table
    for metric in ["eo", "dp"]:
        key = f"abl_no_auditor_{metric}"
        print(f"\n  -- Ablation: {key} (GRL only; auditor disabled) --")
        seed_results = []
        for s in SEEDS_FINAL:
            print(f"    seed {s} ...")
            r = run_ablation(d, cfg, metric, "no_auditor", seed=s)
            seed_results.append(r)
        ds_results[key] = aggregate_seeds(seed_results)

    for metric in ["eo", "dp"]:
        key  = f"agad_{metric}_tuned"
        hp   = BEST_PARAMS[ds_key][metric]
        sm   = SCORE_MODE[ds_key][metric]
        fw   = FG_CKPT_WEIGHT[ds_key][metric]
        mce  = MIN_CKPT_EPOCH[ds_key][metric]
        print(f"\n  -- AGAD-{metric.upper()} (Auditor+GRL, full) --")
        print(f"     hparams  : {hp}")
        seed_results = []
        for s in SEEDS_FINAL:
            print(f"    seed {s} ...")
            r = run_agad_with_hparams(d, cfg, metric, hp, seed=s,
                                       verbose=True, epoch_verbose=True,
                                       score_mode=sm, fg_ckpt_weight=fw,
                                       min_ckpt_epoch=mce,
                                       disable_auditor=False)
            seed_results.append(r)
        ds_results[key] = aggregate_seeds(seed_results)

    all_results[ds_key] = ds_results

    # ── Two-table per-dataset summary (EO table + DP table, AGAD last) ────
    print_two_tables(ds_key, ds_results)

    # ── Ablation analysis ──────────────────────────────────────────────────
    print(f"\n  Ablation — Auditor+GRL (full) vs GRL-only (no_auditor):")
    for metric in ["eo", "dp"]:
        full_wc = ds_results[f"agad_{metric}_tuned"][f"wc_{metric}"]
        abl_wc  = ds_results[f"abl_no_auditor_{metric}"][f"wc_{metric}"]
        delta   = full_wc - abl_wc
        label   = "AUDITOR HELPS ✓" if delta < 0 else "auditor neutral/hurts"
        print(f"    WC-{metric.upper()}  full={full_wc:.4f}  grl_only={abl_wc:.4f}  "
              f"delta={delta:+.5f}  ({label})")

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

run_time = time.time() - t_phase


# ════════════════════════════════════════════════════════════════════════════
#  PLOTS
# ════════════════════════════════════════════════════════════════════════════

print_section("PLOTS")

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "figure.dpi": 150,
    "axes.spines.top": False, "axes.spines.right": False,
})


def fig1_bar_fairness():
    """Bar chart: WC fairness gaps across all methods (SGPen removed)."""
    methods = COMPARISON_METHODS
    fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=False)
    fig.suptitle("Figure 1: Worst-Case Fairness Gaps — All Methods\n(lower is better)",
                 fontsize=13, fontweight="bold", y=1.01)
    for col, ds_key in enumerate(RUN_DATASETS):
        for row, metric in enumerate(["eo", "dp"]):
            ax = axes[row][col]; wc_key = f"wc_{metric}"
            means  = [all_results[ds_key][m][wc_key]          for m in methods]
            stds   = [all_results[ds_key][m][wc_key + "_std"] for m in methods]
            colors = [PALETTE.get(m, "#aaaaaa") for m in methods]
            x = np.arange(len(methods))
            bars = ax.bar(x, means, yerr=stds, color=colors, alpha=0.85,
                          capsize=3, error_kw={"linewidth": 1.2})
            for i, m in enumerate(methods):
                if "agad" in m:
                    bars[i].set_edgecolor("black"); bars[i].set_linewidth(1.5)
            ax.set_xticks(x)
            ax.set_xticklabels([METHOD_LABELS.get(m, m) for m in methods],
                               rotation=55, ha="right", fontsize=6)
            ax.set_title(f"{DS_LABELS[ds_key]}\nWC-{'EO' if metric=='eo' else 'DP'}",
                         fontsize=9)
            ax.set_ylabel("Gap" if col == 0 else "", fontsize=9)
            ax.yaxis.grid(True, linestyle="--", alpha=0.4); ax.set_axisbelow(True)
    plt.tight_layout()
    path = f"{PLOT_DIR}/fig1_wc_fairness_bars.png"
    plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
    print(f"  Saved -> {path}")


def fig2_ablation_bars():
    """2-bar ablation: Full AGAD (Auditor+GRL) vs GRL-only (no_auditor)."""
    fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharey=False)
    fig.suptitle(
        "Figure 2: Ablation Study — Full AGAD (Auditor+GRL) vs GRL Only\n"
        "(lower WC = better; Full AGAD outlined in black)",
        fontsize=13, fontweight="bold", y=1.01)

    colors_eo = ["#2dc653", "#fde68a"]
    colors_dp = ["#1a7c34", "#d97706"]

    for col, ds_key in enumerate(RUN_DATASETS):
        for row, metric in enumerate(["eo", "dp"]):
            ax = axes[row][col]
            methods_abl = ABLATION_EO_METHODS if metric == "eo" else ABLATION_DP_METHODS
            colors_abl  = colors_eo           if metric == "eo" else colors_dp
            wc_key = f"wc_{metric}"
            means = [all_results[ds_key][m][wc_key]          for m in methods_abl]
            stds  = [all_results[ds_key][m][wc_key + "_std"] for m in methods_abl]
            x = np.arange(len(methods_abl))
            bars = ax.bar(x, means, yerr=stds, color=colors_abl, alpha=0.85,
                          capsize=3, error_kw={"linewidth": 1.2})
            bars[0].set_edgecolor("black"); bars[0].set_linewidth(2.0)
            ax.set_xticks(x)
            ax.set_xticklabels(
                [ABLATION_LABELS.get(m, m) for m in methods_abl],
                rotation=0, ha="center", fontsize=9)
            ax.set_title(f"{DS_LABELS[ds_key]}\nWC-{metric.upper()} Ablation", fontsize=9)
            ax.set_ylabel("WC Gap" if col == 0 else "", fontsize=9)
            ax.yaxis.grid(True, linestyle="--", alpha=0.4); ax.set_axisbelow(True)

    from matplotlib.patches import Patch
    legend_elems = [
        Patch(facecolor="#2dc653", edgecolor="black", linewidth=2,
              label="Full AGAD (Auditor+GRL)"),
        Patch(facecolor="#fde68a", label="GRL only (no Auditor)"),
    ]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2,
               fontsize=10, bbox_to_anchor=(0.5, -0.04))
    plt.tight_layout()
    path = f"{PLOT_DIR}/fig2_ablation_bars.png"
    plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
    print(f"  Saved -> {path}")


def fig3_acc_fairness_pareto():
    """
    Accuracy vs WC-Fairness scatter with Pareto frontier.
    All methods unconstrained — shown as-is, tradeoff visible.
    Moved to end as requested.
    """
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=False)
    fig.suptitle(
        "Figure 3: Accuracy vs WC-Fairness Pareto Tradeoff\n"
        "(lower-left = ideal; Pareto frontier dashed; ★ = AGAD)",
        fontsize=13, fontweight="bold", y=1.01)

    for col, ds_key in enumerate(RUN_DATASETS):
        for row, metric in enumerate(["eo", "dp"]):
            ax = axes[row][col]
            wc_key = f"wc_{metric}"
            frontier = pareto_frontier(all_results[ds_key], wc_key, "acc")

            for m in COMPARISON_METHODS:
                r = all_results[ds_key][m]
                c  = PALETTE.get(m, "#aaaaaa")
                mk = "*" if "agad" in m else "o"
                sz = 200 if "agad" in m else 60
                ax.scatter(r["acc"], r[wc_key], color=c,
                           marker=mk, s=sz, zorder=4,
                           label=METHOD_LABELS.get(m, m),
                           edgecolors="black" if "agad" in m else "none",
                           linewidths=1.2)
                ax.errorbar(r["acc"], r[wc_key],
                            xerr=r.get("acc_std", 0),
                            yerr=r.get(wc_key + "_std", 0),
                            fmt="none", color=c, alpha=0.3, linewidth=0.8)

            # Draw Pareto frontier line
            frontier_pts = sorted(
                [(all_results[ds_key][m]["acc"], all_results[ds_key][m][wc_key])
                 for m in frontier
                 if m in COMPARISON_METHODS],
                key=lambda p: p[0])
            if len(frontier_pts) >= 2:
                fx, fy = zip(*frontier_pts)
                ax.plot(fx, fy, "k--", linewidth=1.0, alpha=0.5, zorder=2,
                        label="Pareto frontier")

            ax.set_title(f"{DS_LABELS[ds_key]}\nAcc vs WC-{metric.upper()}", fontsize=9)
            ax.set_xlabel("Accuracy" if row == 1 else "", fontsize=8)
            ax.set_ylabel(f"WC-{metric.upper()}" if col == 0 else "", fontsize=8)
            ax.yaxis.grid(True, linestyle="--", alpha=0.3)
            ax.xaxis.grid(True, linestyle="--", alpha=0.3)
            ax.set_axisbelow(True)

    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=6,
               fontsize=6, bbox_to_anchor=(0.5, -0.06))
    plt.tight_layout()
    path = f"{PLOT_DIR}/fig3_pareto_scatter.png"
    plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
    print(f"  Saved -> {path}")


fig1_bar_fairness()
fig2_ablation_bars()
fig3_acc_fairness_pareto()   # Pareto curve LAST as requested


# ════════════════════════════════════════════════════════════════════════════
#  GLOBAL SUMMARY — two tables (EO + DP), AGAD as last rows
# ════════════════════════════════════════════════════════════════════════════

print_section("GLOBAL SUMMARY — EO TABLE (all datasets)")

row_order_global = [
    "vanilla", "adv_eo", "adv_dp",
    "fairlearn_eo", "fairlearn_dp",
    "prejudice_rem_eo", "prejudice_rem_dp",
    "abl_no_auditor_eo", "abl_no_auditor_dp",
    "agad_eo_tuned", "agad_dp_tuned",   # AGAD last
]

def fmt_cell(r, k, w=18):
    v = r.get(k, float("nan")); s = r.get(k+"_std", 0.0)
    cell = f"{v:.4f}±{s:.4f}" if not np.isnan(v) else "nan"
    return f"{cell:>{w}}"

hdr = f"  {'Dataset':<22}  {'Method':<28}  {'WC-EO':>18}  {'FG-EO':>18}  {'Acc':>12}  {'AUC':>12}"
sep = "  " + "─" * 100
print(hdr); print(sep)
for ds_key in RUN_DATASETS:
    rows = [m for m in row_order_global if m in all_results[ds_key]]
    for i, m in enumerate(rows):
        r        = all_results[ds_key][m]
        ds_label = DS_LABELS[ds_key] if i == 0 else ""
        flag     = " ←★" if "agad" in m else ""
        print(f"  {ds_label:<22}  {METHOD_LABELS.get(m,m):<28}"
              f"{fmt_cell(r,'wc_eo')}{fmt_cell(r,'fg_eo')}"
              f"{fmt_cell(r,'acc',12)}{fmt_cell(r,'auc',12)}{flag}")
    print(sep)

print_section("GLOBAL SUMMARY — DP TABLE (all datasets)")

hdr = f"  {'Dataset':<22}  {'Method':<28}  {'WC-DP':>18}  {'FG-DP':>18}  {'Acc':>12}  {'AUC':>12}"
print(hdr); print(sep)
for ds_key in RUN_DATASETS:
    rows = [m for m in row_order_global if m in all_results[ds_key]]
    for i, m in enumerate(rows):
        r        = all_results[ds_key][m]
        ds_label = DS_LABELS[ds_key] if i == 0 else ""
        flag     = " ←★" if "agad" in m else ""
        print(f"  {ds_label:<22}  {METHOD_LABELS.get(m,m):<28}"
              f"{fmt_cell(r,'wc_dp')}{fmt_cell(r,'fg_dp')}"
              f"{fmt_cell(r,'acc',12)}{fmt_cell(r,'auc',12)}{flag}")
    print(sep)


# ════════════════════════════════════════════════════════════════════════════
#  PARETO FRONTIER SUMMARY  (at the end, as requested)
# ════════════════════════════════════════════════════════════════════════════

print_section("PARETO FRONTIER SUMMARY (accuracy-fairness tradeoff, unconstrained)")
for ds_key in RUN_DATASETS:
    print(f"\n  {DS_LABELS[ds_key]}:")
    for metric in ["eo", "dp"]:
        frontier = pareto_frontier(all_results[ds_key], f"wc_{metric}", "acc")
        agad_key = f"agad_{metric}_tuned"
        on = agad_key in frontier
        print(f"    WC-{metric.upper()} frontier : {[METHOD_LABELS.get(m,m) for m in frontier]}")
        print(f"    AGAD-{metric.upper()} on frontier: {'YES ✓' if on else 'NO ✗'}")


# ════════════════════════════════════════════════════════════════════════════
#  AUDITOR CONTRIBUTION SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print_section("AUDITOR CONTRIBUTION SUMMARY")
print("  Full AGAD (Auditor+GRL) vs GRL-only (no_auditor):")
print("  negative delta = auditor reduces worst-case unfairness")
print()
for ds_key in RUN_DATASETS:
    print(f"  {DS_LABELS[ds_key]}:")
    for metric in ["eo", "dp"]:
        full_wc = all_results[ds_key][f"agad_{metric}_tuned"][f"wc_{metric}"]
        abl_wc  = all_results[ds_key][f"abl_no_auditor_{metric}"][f"wc_{metric}"]
        delta   = full_wc - abl_wc
        verdict = "AUDITOR HELPS ✓" if delta < 0 else "auditor neutral/hurts"
        print(f"    {metric.upper()}  full={full_wc:.4f}  grl_only={abl_wc:.4f}  "
              f"delta={delta:+.5f}  {verdict}")
    print()


# ════════════════════════════════════════════════════════════════════════════
#  SAVE RESULTS
# ════════════════════════════════════════════════════════════════════════════

def clean(obj):
    if isinstance(obj, dict):          return {k: clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [clean(v) for v in obj]
    if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)): return None
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray):   return obj.tolist()
    return obj

payload = {
    "results"     : all_results,
    "best_params" : BEST_PARAMS,
    "runtime"     : {"total_min": run_time / 60},
    "seeds"       : SEEDS_FINAL,
    "version"     : "v8",
    "architecture": {
        "sgpen"   : "FULLY REMOVED — no gamma_sg, no FrozenMaskManager, no penalty term, not even as a baseline",
        "auditor" : "Subgroup discovery drives adaptive_lambda for GRL",
        "grl"     : "Intersection adversary with GRL; always active",
        "ablation": "Three conditions: vanilla | GRL-only (disable_auditor=True) | full AGAD (Auditor+GRL)",
    },
    "output_format": (
        "Two tables per dataset and globally: EO table (WC-EO, FG-EO, Acc, AUC) "
        "and DP table (WC-DP, FG-DP, Acc, AUC). AGAD rows always last for easy comparison."
    ),
    "pareto_note": (
        "All methods unconstrained — Pareto frontier used for honest comparison. "
        "PrejudiceRemover reported straight (may have accuracy tradeoff). "
        "Pareto plots generated at end of run."
    ),
}

out_path = f"{WORK_DIR}/agad_v8_results.json"
with open(out_path, "w") as f:
    json.dump(clean(payload), f, indent=2)

print_section("DONE")
print(f"  Total time   : {run_time/60:.1f} min")
print(f"  Results      -> {out_path}")
print(f"  Plots        -> {PLOT_DIR}/")

  AGAD v8 — SGPen fully removed; 3-condition ablation; clean tables
  Device     : cuda
  Seeds      : [42]
  SGPen      : FULLY REMOVED (no baselines, no loss term, no params)
  Ablations  : vanilla | GRL-only (no_auditor) | full AGAD (Auditor+GRL)
  Output     : two tables per dataset — EO table and DP table, AGAD last row
  Baselines  : all unconstrained — Pareto frontier reported at end

  FINAL EVALUATION  (1 seeds: [42])

----------------------------------------------------------------------
  Dataset: ADULT
----------------------------------------------------------------------
  Train=31258  Val=7815  Test=9769

  -- vanilla --
    seed 42 ...     [Vanilla NN                          ]  WC-EO=0.0762  WC-DP=0.0613  FG-EO=0.0703  FG-DP=0.0563  Acc=0.7413  AUC=0.8481
WC-EO=0.0762  WC-DP=0.0613  Acc=0.7413  AUC=0.8481

  -- adv_eo --
    seed 42 ...     [Adv-EO (static)                     ]  WC-EO=0.0716  WC-DP=0.0581  FG-EO=0.0628  FG-DP=0.0530  Acc=0.7797  AUC=0.8559
WC-EO=0.0716